# Multi-task pretraining

> Pretrain the CNN on 23 OSSL soil properties at once, and compare its frozen layers with those of the single-task Kex and CEC models for Fukushima Kex, CEC and the Cs-137 ratio

In [ ]:
#| default_exp multitask

In [ ]:
#| export
import numpy as np, pandas as pd, torch
from torch.nn import Linear
from fastai.torch_core import set_seed
import fastai.callback.schedule
from soilspecdl.all import get_spectral_dls, get_splitter, spectral_learner, BatchSNV, SpectralCNN
from ossl_fuku_transfer.data import shared_wns
from ossl_fuku_transfer.transfer import model_dir

/Users/franckalbinet/pro/dev/soilspec/ossl-fuku-transfer/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
import matplotlib.pyplot as plt
from fastcore.test import *
from sklearn.metrics import r2_score
from soilspecdata.all import load_ossl
from soilspecdl.all import TensorSpectrum, SpectralCNN, predict, regression_scores
from ossl_fuku_transfer.data import *
from ossl_fuku_transfer.transfer import *
from ossl_fuku_transfer.inside import layer_features, probe

`07_inside_transfer` and `10_cs137` found that frozen layers of the single-task OSSL models, read by a ridge regression, predict Fukushima Kex and the Cs-137 ratio better than the spectra, but only a little better than the same layers with random weights. A model trained on many soil properties at once may learn features that are more general, and so transfer better.

The module part uses small random arrays, so it runs without the data. The sections from *Data* onwards read OSSL and the Fukushima data, and are marked `#| fukudata`.

## Tasks and model

`mt_tasks` lists the OSSL properties used as tasks: the 23 with at least 15,000 labelled MIR spectra, apart from depths, coordinates, coarse fragments and the second organic carbon method. The value says whether the task takes `log1p` of the property, which is done for the skewed, non-negative ones. The OSSL column names give each property's method and unit.

In [ ]:
#| export
mt_tasks = { # OSSL column: True if the task takes `log1p` of the property
    'c.tot_usda.a622_w.pct': True, 'n.tot_usda.a623_w.pct': True, 'oc_usda.c729_w.pct': True,
    'ph.h2o_usda.a268_index': False, 'ph.cacl2_usda.a481_index': False,
    'k.ext_usda.a725_cmolc.kg': True, 'mg.ext_usda.a724_cmolc.kg': True, 'ca.ext_usda.a722_cmolc.kg': True,
    'na.ext_usda.a726_cmolc.kg': True, 'cec_usda.a723_cmolc.kg': True, 'acidity_usda.a795_cmolc.kg': True,
    'clay.tot_usda.a334_w.pct': False, 'silt.tot_usda.c62_w.pct': False, 'sand.tot_usda.c60_w.pct': False,
    'wr.1500kPa_usda.a417_w.pct': True, 'bd_usda.a4_g.cm3': False,
    'fe.dith_usda.a66_w.pct': True, 'al.dith_usda.a65_w.pct': True, 'fe.ox_usda.a60_w.pct': True, 'al.ox_usda.a59_w.pct': True,
    'caco3_usda.a54_w.pct': True, 'ec_usda.a364_ds.m': True, 'p.ext_usda.a652_mg.kg': True}

`mt_targets` turns the properties into task targets: `log1p` where `mt_tasks` says so, after clipping negative values at 0, then standardised to mean 0 and standard deviation 1. Missing values stay NaN:

In [ ]:
#| export
def mt_targets(
    df:pd.DataFrame, # One row per sample, with the columns of `tasks`
    tasks:dict=mt_tasks, # Column: True to take `log1p`
)->tuple: # (Targets as a float32 array with NaN where missing, DataFrame of each task's mean and std)
    "Standardised targets for multi-task training, with `log1p` where `tasks` says so"
    Y = pd.DataFrame({c: np.log1p(df[c].clip(lower=0)) if log else df[c] for c, log in tasks.items()})
    mu, sd = Y.mean(), Y.std()
    return ((Y - mu) / sd).to_numpy(np.float32), pd.DataFrame({'mean': mu, 'std': sd})

In [ ]:
df_syn = pd.DataFrame({'a': [0., 1, 3, np.nan], 'b': [5., 6, -1, 7]})
Y_syn, st_syn = mt_targets(df_syn, {'a': True, 'b': False})
test_eq(np.isnan(Y_syn).sum(), 1)
test_close(np.nanmean(Y_syn, 0), [0, 0])
test_close(st_syn.loc['a', 'mean'], np.log1p([0, 1, 3]).mean())

`masked_mse` is the mean squared error over the targets that are not NaN, so each sample counts for the properties it has. `mt_model` is `SpectralCNN` with one output per task:


In [ ]:
#| export
def masked_mse(
    pred, # Predictions, one column per task
    targ, # Targets, NaN where missing
):
    "Mean squared error over the targets that are not NaN"
    m = ~torch.isnan(targ)
    return ((pred - torch.nan_to_num(targ))**2)[m].mean()

def mt_model(
    n_tasks, # Number of outputs
):
    "`SpectralCNN` with a head of `n_tasks` outputs"
    m = SpectralCNN()
    m.head[-1] = Linear(m.head[-1].in_features, n_tasks)
    return m

In [ ]:
p, t = torch.tensor([[1., 2], [3, 4]]), torch.tensor([[2., float('nan')], [3, 6]])
test_close(masked_mse(p, t), (1 + 0 + 4) / 3)
rng = np.random.default_rng(0)
X_syn = rng.random((8, len(shared_wns)))
xb = BatchSNV()(TensorSpectrum(torch.tensor(X_syn, dtype=torch.float32).unsqueeze(1), wns=shared_wns))
test_eq(mt_model(3)(xb).shape, (8, 3))

`pretrain_mt` trains `mt_model` as `pretrain` trains `SpectralCNN`: a random 10% of the samples for validation, a one-cycle schedule, and the weights and losses saved under `fname`, which later calls load instead of training again. `load_mt` loads saved weights:


In [ ]:
#| export
def load_mt(fname, n_tasks):
    "Load the weights that `pretrain_mt` saved under `fname` into `mt_model(n_tasks)`"
    model = mt_model(n_tasks)
    model.load_state_dict(torch.load(model_dir()/f'{fname}.pth', map_location='cpu'))
    return model

def pretrain_mt(
    X, # Preprocessed OSSL spectra
    Y, # Targets from `mt_targets`, NaN where missing
    fname, # File name for the weights, without extension
    seed=42, # Seed for the validation split and the training
    epochs=20, # Number of one-cycle epochs
    lr=1e-3, # Maximum learning rate
):
    "Train `mt_model` on all tasks of `Y`, or load the weights that an earlier run saved under `fname`"
    fn = model_dir()/f'{fname}.pth'
    if not fn.exists():
        set_seed(seed)
        dls, _ = get_spectral_dls(X, Y, shared_wns, splitter=get_splitter(0.9, 0.1, seed=seed), batch_tfms=[BatchSNV()])
        learn = spectral_learner(dls, mt_model(Y.shape[1]), loss_func=masked_mse, metrics=[])
        with learn.no_bar(), learn.no_logging(): learn.fit_one_cycle(epochs, lr_max=lr)
        torch.save(learn.model.state_dict(), fn)
        pd.DataFrame(learn.recorder.values, columns=learn.recorder.metric_names[1:-1]).to_csv(fn.with_suffix('.csv'), index_label='epoch')
    return load_mt(fname, Y.shape[1])

One epoch on random data checks that training runs with missing targets:

In [ ]:
X_syn = rng.random((80, len(shared_wns)))
Y_syn = rng.normal(size=(80, 3)).astype(np.float32)
Y_syn[rng.random((80, 3)) < 0.5] = np.nan
dls_syn = get_spectral_dls(X_syn, Y_syn, shared_wns, splitter=get_splitter(0.8, 0.2), batch_tfms=[BatchSNV()])[0]
learn_syn = spectral_learner(dls_syn, mt_model(3), loss_func=masked_mse, metrics=[])
with learn_syn.no_bar(), learn_syn.no_logging(): learn_syn.fit_one_cycle(1)
test_eq(np.isfinite(learn_syn.recorder.values[-1]).all(), True)

## Data

All OSSL MIR spectra on the shared grid with at least one task label, with SNV and a Savitzky-Golay first derivative. The number of labelled spectra per task:

In [ ]:
#| fukudata
om = load_ossl().mir(wmin=shared_wns[0], wmax=shared_wns[-1])
props = om.source[list(mt_tasks)].reindex(om.sample_ids)
keep = props.notna().any(axis=1).to_numpy()
Y_mt, stats = mt_targets(props[keep])
pipe = preprocessings()['snv_d1']
Xo_d = pipe.fit_transform(om.spectra[keep])
print(Xo_d.shape)
stats.assign(n=np.isfinite(Y_mt).sum(0)).round(3)

(85184, 1675)


,mean,std,n
c.tot_usda.a622_w.pct,1.281,1.104,80439
n.tot_usda.a623_w.pct,0.241,0.318,80454
oc_usda.c729_w.pct,0.966,0.943,55900
ph.h2o_usda.a268_index,6.380,1.327,59408
ph.cacl2_usda.a481_index,5.922,1.388,52900
k.ext_usda.a725_cmolc.kg,0.406,0.379,57674
mg.ext_usda.a724_cmolc.kg,1.369,0.915,57093
ca.ext_usda.a722_cmolc.kg,2.366,1.309,57085
na.ext_usda.a726_cmolc.kg,0.383,0.911,57075
cec_usda.a723_cmolc.kg,2.705,0.894,57062


## Pretraining

20 epochs at a maximum learning rate of 10⁻³, as for the single-task models. The last epoch's losses, the mean squared error over the standardised targets that are present:

In [ ]:
#| fukudata
mt = pretrain_mt(Xo_d, Y_mt, 'ossl_multitask_snv_d1')
pd.read_csv(model_dir()/'ossl_multitask_snv_d1.csv', index_col='epoch').iloc[-1].round(4).to_dict()

/Users/franckalbinet/pro/dev/soilspec/ossl-fuku-transfer/.venv/lib/python3.13/site-packages/fastai/torch_core.py:140: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  t = torch.as_tensor(x, **kwargs)


{'train_loss': 0.1218, 'valid_loss': 0.1063}

The R² of each task on the validation samples. For Kex, the single-task model had a validation R² of 0.79 (`03_kex`), on a different random 10% of OSSL:

In [ ]:
#| fukudata
va = get_splitter(0.9, 0.1, seed=42)(Xo_d, Y_mt)[1]
P = predict(mt, Xo_d[va], shared_wns)
ok = np.isfinite(Y_mt[va])
pd.Series({c: r2_score(Y_mt[va][ok[:, j], j], P[ok[:, j], j]) for j, c in enumerate(mt_tasks)}, name='R²').round(3).to_frame()

,R²
c.tot_usda.a622_w.pct,0.986
n.tot_usda.a623_w.pct,0.956
oc_usda.c729_w.pct,0.979
ph.h2o_usda.a268_index,0.896
ph.cacl2_usda.a481_index,0.919
k.ext_usda.a725_cmolc.kg,0.751
mg.ext_usda.a724_cmolc.kg,0.896
ca.ext_usda.a722_cmolc.kg,0.938
na.ext_usda.a726_cmolc.kg,0.796
cec_usda.a723_cmolc.kg,0.957


## Fukushima

Three Fukushima targets: Kex and CEC as `log1p` of cmolc/kg, scored with `scores`, and the Cs-137 ratio of `10_cs137`, scored with `regression_scores`. All use the Cs137 study's test set. The models: the multi-task model, the single-task Kex and CEC models, the two single-task models' features side by side, and a model with random weights:


In [ ]:
#| fukudata
fuku = load_fuku()
src = fuku.source
ref = fuku_path()/'reference'
ratio = pd.concat([pd.read_csv(ref/f'cs137_ratio_{s}.csv', usecols=['ID', 'exCs137_totalCs137']) for s in ('train', 'test')])
ratio = ratio.assign(ID=ratio.ID.astype(str)).set_index('ID')['exCs137_totalCs137']
ids = fuku.sample_ids
targets = {'Kex': (np.log1p(src['soil_ex_K2O'].reindex(ids) / mg_k2o_per_cmolc).to_numpy(), scores),
           'CEC': (np.log1p(src['soil_CEC'].reindex(ids)).to_numpy(), scores),
           'Cs ratio': (ratio.reindex(ids).to_numpy(), regression_scores)}
is_test = np.isin(ids, cs137_split()[1])
Xf_d = pipe.transform(fuku.spectra)
set_seed(0)
models = {'multi-task': mt, 'Kex': load_pretrained('ossl_kex_snv_d1'), 'CEC': load_pretrained('ossl_cec_snv_d1'), 'random': SpectralCNN()}
F = {m: layer_features(model, Xf_d) for m, model in models.items()}
F['Kex + CEC'] = {k: np.hstack([F['Kex'][k], F['CEC'][k]]) for k in F['Kex']}
{t: (int((np.isfinite(y) & ~is_test).sum()), int((np.isfinite(y) & is_test).sum())) for t, (y, _) in targets.items()}

{'Kex': (1047, 341), 'CEC': (1046, 340), 'Cs ratio': (1048, 341)}

A ridge `probe` at each layer, trained on the training samples with a value and scored on the test samples with a value. The test R² for each target:

In [ ]:
#| fukudata
rows = []
for t, (y, sf) in targets.items():
    tr, te = np.isfinite(y) & ~is_test, np.isfinite(y) & is_test
    for m, Fm in F.items():
        for k in Fm: rows.append({'target': t, 'model': m, 'layer': k, **probe(Fm[k][tr], y[tr], Fm[k][te], y[te], score_fn=sf)})
lay = pd.DataFrame(rows)
lay.pivot_table(index='layer', columns=['target', 'model'], values='R²', sort=False).round(3)

target         Kex                       ...                               
model   multi-task    Kex    CEC random  ...    Kex    CEC random Kex + CEC
layer                                    ...                               
input        0.332  0.332  0.332  0.332  ...  0.537  0.537  0.537     0.533
block 1      0.379  0.378  0.374  0.375  ...  0.594  0.589  0.566     0.586
block 2      0.430  0.403  0.402  0.388  ...  0.613  0.607  0.597     0.618
block 3      0.404  0.396  0.401  0.401  ...  0.613  0.614  0.590     0.626
block 4      0.429  0.429  0.403  0.366  ...  0.641  0.635  0.613     0.648
block 5      0.408  0.371  0.375  0.363  ...  0.625  0.623  0.584     0.658
pooled       0.254  0.253  0.261  0.311  ...  0.479  0.544  0.482     0.564

[7 rows x 15 columns]

The best layer of each model for each target:

In [ ]:
#| fukudata
best = lay.loc[lay.groupby(['target', 'model'], sort=False)['R²'].idxmax(), ['target', 'model', 'layer', 'R²', 'r']]
best.set_index(['target', 'model']).round(3)

layer     R²      r
target   model                            
Kex      multi-task  block 2  0.430  0.656
         Kex         block 4  0.429  0.657
         CEC         block 4  0.403  0.637
         random      block 3  0.401  0.634
         Kex + CEC   block 4  0.442  0.666
CEC      multi-task  block 4  0.737  0.859
         Kex         block 4  0.746  0.864
         CEC         block 4  0.733  0.856
         random      block 3  0.722  0.851
         Kex + CEC   block 4  0.747  0.865
Cs ratio multi-task  block 4  0.624  0.796
         Kex         block 4  0.641  0.805
         CEC         block 4  0.635  0.801
         random      block 4  0.613  0.789
         Kex + CEC   block 5  0.658  0.814

## Few training samples

Transfer helped most with few training samples in `08_learning_curve` and `10_cs137`. Ridge regressions on block 4 of each model, trained on 200 and on 400 random training samples, five draws each, and scored on the same test samples. The mean test R²:

In [ ]:
#| fukudata
rows = []
for t, (y, sf) in targets.items():
    tr_all, te = np.flatnonzero(np.isfinite(y) & ~is_test), np.isfinite(y) & is_test
    for n in (200, 400):
        for s in range(5):
            sub = np.random.RandomState(s).choice(tr_all, n, replace=False)
            for m, Fm in F.items(): rows.append({'target': t, 'n': n, 'seed': s, 'model': m, **probe(Fm['block 4'][sub], y[sub], Fm['block 4'][te], y[te], score_fn=sf)})
few = pd.DataFrame(rows)
few.groupby(['target', 'n', 'model'], sort=False)['R²'].mean().unstack('model').round(3)

model         multi-task    Kex    CEC  random  Kex + CEC
target   n                                               
Kex      200       0.251  0.233  0.237   0.187      0.230
         400       0.327  0.325  0.308   0.265      0.324
CEC      200       0.659  0.652  0.662   0.646      0.658
         400       0.698  0.696  0.693   0.656      0.705
Cs ratio 200       0.497  0.490  0.486   0.426      0.506
         400       0.563  0.578  0.568   0.520      0.588

## Findings

The multi-task model learns all 23 properties on OSSL. Its validation R² ranges from 0.62 for extractable P to 0.99 for total carbon, and is 0.90 or more for 12 of the 23 tasks. For Kex it is 0.75, slightly below the single-task model's 0.79, on a different random validation set.

Its frozen layers transfer to Fukushima no better than those of the single-task models. The best layer of each model gives these test R²:

- Kex: 0.43 for the multi-task model (block 2) and the Kex model (block 4), and 0.44 for the Kex and CEC models together;
- CEC: 0.74 for the multi-task model, 0.75 for the Kex model, and 0.75 for the Kex and CEC models together;
- Cs-137 ratio: 0.62 for the multi-task model, 0.64 for the Kex model, and 0.66 for the Kex and CEC models together (block 5).

With 341 test samples, one R² has a sampling standard deviation of about 0.04, so these differences are within noise. With 200 and 400 training samples, block 4 of the multi-task model scores within 0.03 of the single-task models for every target.

At their best layer with all training samples, the pretrained models beat random weights by 0.00 to 0.05; the CEC model's best layer for Kex is level with random. With 200 or 400 training samples, block 4 of every pretrained model beats random weights by 0.04 to 0.08 for Kex and the Cs-137 ratio.

Which OSSL properties a model learns makes little difference to how well its frozen layers serve Fukushima. With these pretraining data, the transfer seems limited by how far Fukushima spectra sit from OSSL (`07_inside_transfer`), not by the pretraining objective.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()